# SI Fig 3: Tara (Ocean 16S) — Hyperparameter comparison

Compare consensus groupings across hyperparameter settings for Tara 16S data.

In [ ]:
import sys, os, time
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.autonotebook import tqdm
import plotly.graph_objects as go
import IPython

from utils.models import FC_Gumbelpredictor
from dataset import TaraDataset

plt.rcParams['axes.prop_cycle'] = plt.cycler(color=plt.cm.Set1.colors)
plt.rcParams['figure.figsize'] = [3, 2]
plt.rcParams['figure.dpi'] = 200
plt.rcParams['svg.fonttype'] = 'none'

device = torch.device('cpu')

## Load models

In [ ]:
masterroot = './trained_models'

subroots = [
    'gumbel_241204_113035_N-128_L-2_DO-0.05_LR-1e-2_TAU-2._0.9999_0.1_SCH-0.9999_OPT-adam_ACT-gelu_NENS-15_PREC-0_Nsteps-50000',
    'gumbel_241204_113036_N-128_L-2_DO-0.05_LR-1e-2_TAU-2._0.999_0.1_SCH-0.9999_OPT-adam_ACT-gelu_NENS-15_PREC-0_Nsteps-50000',
    'gumbel_241204_113036_N-128_L-2_DO-0._LR-1e-2_TAU-2._0.9999_0.1_SCH-0.9999_OPT-adam_ACT-gelu_NENS-15_PREC-0_Nsteps-50000',
    'gumbel_241204_113036_N-128_L-2_DO-0._LR-1e-2_TAU-2._0.999_0.1_SCH-0.9999_OPT-adam_ACT-gelu_NENS-15_PREC-0_Nsteps-50000',
    'gumbel_241204_113036_N-128_L-2_DO-0._LR-1e-3_TAU-2._0.9999_0.1_SCH-0.9999_OPT-adam_ACT-gelu_NENS-15_PREC-0_Nsteps-50000',
    'gumbel_241204_113036_N-128_L-2_DO-0._LR-1e-3_TAU-2._0.999_0.1_SCH-0.9999_OPT-adam_ACT-gelu_NENS-15_PREC-0_Nsteps-50000',
    'gumbel_241204_113036_N-128_L-2_DO-0._LR-1e-4_TAU-2._0.9999_0.1_SCH-0.9999_OPT-adam_ACT-gelu_NENS-15_PREC-0_Nsteps-50000',
    'gumbel_241204_113036_N-128_L-2_DO-0._LR-1e-4_TAU-2._0.999_0.1_SCH-0.9999_OPT-adam_ACT-gelu_NENS-15_PREC-0_Nsteps-50000',
    'gumbel_241204_133636_N-128_L-2_DO-0.05_LR-1e-3_TAU-2._0.999_0.1_SCH-0.9999_OPT-adam_ACT-gelu_NENS-15_PREC-0_Nsteps-50000',
    'gumbel_241204_133722_N-128_L-2_DO-0.05_LR-1e-3_TAU-2._0.9999_0.1_SCH-0.9999_OPT-adam_ACT-gelu_NENS-15_PREC-0_Nsteps-50000',
    'gumbel_241204_133808_N-128_L-2_DO-0.05_LR-1e-4_TAU-2._0.999_0.1_SCH-0.9999_OPT-adam_ACT-gelu_NENS-15_PREC-0_Nsteps-50000',
    'gumbel_241204_133909_N-128_L-2_DO-0.05_LR-1e-4_TAU-2._0.9999_0.1_SCH-0.9999_OPT-adam_ACT-gelu_NENS-15_PREC-0_Nsteps-50000'
]

loss_df = []
dataset_train = None
t0 = time.time()
pb = tqdm(total=len(subroots))

for sr in subroots:
    root = os.path.join(masterroot, sr)
    for file in os.listdir(root):
        npcs = int(file.split('_')[0].split('-')[-1])
        ensidx = int(file.split('_')[1].split('-')[-1].split('.')[0])

        x = torch.load(os.path.join(root, file), map_location=device)
        tauparams = {'tau_' + k: v for k, v in x.get('tau_hparams', {}).items()}

        if dataset_train is None:
            train_kwargs = dict(x['train_dataset_kwargs'])
            train_kwargs['device'] = device
            dataset_train = TaraDataset(**train_kwargs)

        pred = np.asarray(x.get('pred_train', []))
        targ = np.asarray(x.get('target_train', []))
        test_pred = np.asarray(x.get('pred_test', []))
        test_targ = np.asarray(x.get('target_test', []))

        logits = np.asarray(x.get('proj_logits', np.nan))
        prob = np.asarray(x.get('proj_det', np.nan))

        loss_df.append({
            'npcs': npcs, 'ensidx': ensidx,
            'tau': x.get('curr_tau', np.nan),
            'train_idx': np.asarray(x.get('train_idx_split', x.get('train_dataset_kwargs', {}).get('train_idx', []))),
            'test_idx': np.asarray(x.get('test_idx_split', x.get('test_dataset_kwargs', {}).get('train_idx', []))),
            'train_loss': x.get('train_losses', []), 'test_loss': x.get('test_losses', []),
            'train_preds': pred.squeeze() if pred.size else pred, 'train_targets': targ.squeeze() if targ.size else targ,
            'test_preds': test_pred.squeeze() if test_pred.size else test_pred, 'test_targets': test_targ.squeeze() if test_targ.size else test_targ,
            'r2_test': x.get('r2_test', np.nan),
            'r2_train': x.get('r2_train', np.nan),
            'proj': np.asarray(x.get('proj', np.nan)),
            'proj_det': prob,
            'logits': logits,
            'lr': x.get('model_kwargs', {}).get(1, {}).get('optimizer_hparams', {}).get('LR', np.nan),
            'dropout': x.get('model_kwargs', {}).get(1, {}).get('network_hparams', {}).get('dropout', np.nan),
            **tauparams
        })
    pb.update(1)

print(f'Time elapsed: {time.time() - t0:.1f}s')
loss_df = pd.DataFrame(loss_df)

names = np.asarray([x.split(';')[-1].lower() for x in dataset_train.df.columns][2:])
abd = dataset_train.inputs.detach().cpu().numpy()

print('loss_df shape:', loss_df.shape)
print('n species:', len(names))

## Configure hyperparameter sweep

In [ ]:
hparam_cols = ['lr', 'tau_relax_rate', 'dropout', 'npcs']
available_hparams = {k: np.sort(loss_df[k].unique()) for k in hparam_cols}

print('Available values from loss_df:')
for k, v in available_hparams.items():
    print(k, '->', v)

npc_fixed = 2
n_ensemble_keep = 5
membership_certainty_threshold = 0.0

vary_keys = ['lr', 'tau_relax_rate', 'dropout']
baseline = {k: available_hparams[k][0] for k in vary_keys}

print('\nFixed params:')
print('npc_fixed =', npc_fixed)
print('n_ensemble_keep =', n_ensemble_keep)
print('membership_certainty_threshold =', membership_certainty_threshold)
print('\nBaseline varying params:', baseline)

In [ ]:
setting_list = [baseline.copy()]
for k in vary_keys:
    for val in available_hparams[k]:
        h = baseline.copy()
        h[k] = val
        setting_list.append(h)

seen = set()
setting_list_unique = []
for h in setting_list:
    key = tuple((k, h[k]) for k in vary_keys)
    if key not in seen:
        seen.add(key)
        setting_list_unique.append(h)
setting_list = setting_list_unique

setting_labels = [f"lr={h['lr']}, tau={h['tau_relax_rate']}, do={h['dropout']}" for h in setting_list]

print('n settings:', len(setting_list))
for i, h in enumerate(setting_list):
    print(i, h)

## Compute consensus groupings per setting

In [ ]:
def get_consensus_ids_tara(hparams, npc=4, n_ensemble_keep=6):
    mask = ((loss_df.npcs == npc) &
            (loss_df.lr == hparams['lr']) &
            (loss_df.tau_relax_rate == hparams['tau_relax_rate']) &
            (loss_df.dropout == hparams['dropout']))

    test_r2 = loss_df.loc[mask, 'r2_test'].values
    if len(test_r2) == 0:
        return None, None

    ensembled_sorted = loss_df.loc[mask, 'ensidx'].values[np.argsort(test_r2)[::-1]]
    ensembled_sorted = ensembled_sorted[:n_ensemble_keep]

    all_projs = np.zeros((len(ensembled_sorted), len(names), npc))
    for e, ens_idx in enumerate(ensembled_sorted):
        proj = loss_df.loc[mask & (loss_df.ensidx == ens_idx), 'proj_det'].values[0].copy()
        proj[np.arange(len(proj)), np.argmax(proj, axis=1)] = 1
        proj[proj < 1] = 0

        corrs = np.zeros(npc)
        all_projs[e] = proj
        for i in range(npc):
            intragp_abd = abd * proj[None, :, i]
            corrs[i] = np.mean(np.mean(intragp_abd, axis=1))

        group_sort = np.argsort(corrs)[::-1]
        all_projs[e] = all_projs[e][:, group_sort]

    mean_membership = np.mean(all_projs, axis=0)
    consensus_ids = np.argmax(mean_membership, axis=1)
    group_certainties = np.max(mean_membership, axis=1)
    spec_keep = np.where(group_certainties > membership_certainty_threshold)[0]
    return consensus_ids, spec_keep

In [ ]:
assignment_lists = []
spec_keep_by_setting = []
setting_labels_kept = []

for hparams in setting_list:
    ids, spec_keep = get_consensus_ids_tara(hparams, npc=npc_fixed, n_ensemble_keep=n_ensemble_keep)
    if ids is None:
        print('skip (no runs):', hparams)
        continue

    assignment_lists.append(ids)
    spec_keep_by_setting.append(spec_keep)
    setting_labels_kept.append(f"lr={hparams['lr']}, tau={hparams['tau_relax_rate']}, do={hparams['dropout']}")

if len(spec_keep_by_setting) == 0:
    raise RuntimeError('No valid settings found for current fixed params.')

spec_keep = spec_keep_by_setting[0]
for s in spec_keep_by_setting[1:]:
    spec_keep = np.intersect1d(spec_keep, s)

print('n valid settings:', len(assignment_lists))
print('n species kept across settings:', len(spec_keep))

assignment_lists_shifted = []
for ids in assignment_lists:
    ids_shifted = ids.copy()
    if len(assignment_lists_shifted) > 0:
        ids_shifted = ids_shifted + assignment_lists_shifted[-1].max() + 1
    assignment_lists_shifted.append(ids_shifted)

## Sankey diagram across hyperparameter settings

In [ ]:
tnames = ['nitrosopumilaceae', 'microtrichaceae', 'sup05_cluster', 'nitrincolaceae',
          'colwellia', 'ns9_marine_group', 'candidatus_scalindua', 'halomonas',
          'oleibacter', 'polaribacter', 'candidatus_kaiserbacteria']
tnames_set = set([t.lower() for t in tnames])

names_plot = names[spec_keep]
species_mean_abd = dataset_train.inputs.mean(axis=0).detach().cpu().numpy()[spec_keep]

assign_mat = np.stack([ids[spec_keep] for ids in assignment_lists], axis=0)
is_stable = np.all(assign_mat == assign_mat[0:1, :], axis=0)
is_changed = ~is_stable

species_colors = []
for nm, changed in zip(names_plot, is_changed):
    if nm.lower() in tnames_set:
        c = 'royalblue' if changed else 'navy'
    else:
        c = 'lightgray' if changed else 'dimgray'
    species_colors.append(c)

srcs, trgs, link_names, link_colors_all, widths = [], [], [], [], []

for i in range(len(assignment_lists_shifted) - 1):
    srcs += assignment_lists_shifted[i][spec_keep].tolist()
    trgs += assignment_lists_shifted[i + 1][spec_keep].tolist()
    link_names += names_plot.tolist()
    link_colors_all += species_colors
    widths += species_mean_abd.tolist()

node_values = np.unique(np.concatenate([ids[spec_keep] for ids in assignment_lists_shifted]))
node_map = {v: i for i, v in enumerate(node_values)}
source = [node_map[s] for s in srcs]
target = [node_map[t] for t in trgs]

node_labels = []
for i, ids in enumerate(assignment_lists_shifted):
    row_ids = ids[spec_keep]
    row_min = row_ids.min()
    for g in np.unique(row_ids):
        node_labels.append(f"{setting_labels_kept[i]} | group {g - row_min}")

fig = go.Figure(data=[go.Sankey(
    node=dict(pad=15, thickness=20,
              line=dict(color='black', width=0.5),
              label=node_labels, color='lightblue'),
    link=dict(source=source, target=target,
              value=widths, label=link_names,
              line=dict(width=0), color=link_colors_all)
)])

fig.update_layout(height=750, width=1300)
fig.show()

if input('save? ').strip().lower() == 'save':
    fig.write_image(f'./figures/SI_tara_hparams_sankey_npc{npc_fixed}.svg',
                    format='svg', width=1300, height=750)

## R² distribution per hyperparameter setting

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(max(6, 1.2 * len(setting_list)), 3), dpi=200)
jitter = 0.10

for i, hparams in enumerate(setting_list):
    mask = ((loss_df.npcs == npc_fixed) &
            (loss_df.lr == hparams['lr']) &
            (loss_df.tau_relax_rate == hparams['tau_relax_rate']) &
            (loss_df.dropout == hparams['dropout']))

    r2_vals = loss_df.loc[mask, 'r2_test'].values.astype(float)
    if len(r2_vals) == 0:
        continue

    order = np.argsort(r2_vals)[::-1]
    top_idx = order[:min(n_ensemble_keep, len(order))]
    rest_idx = order[min(n_ensemble_keep, len(order)):]

    if len(rest_idx) > 0:
        x_rest = i + np.random.uniform(-jitter, jitter, size=len(rest_idx))
        ax.scatter(x_rest, r2_vals[rest_idx], s=14, alpha=0.45, color='gray', lw=0,
                   label='other' if i == 0 else None)

    x_top = i + np.random.uniform(-jitter, jitter, size=len(top_idx))
    ax.scatter(x_top, r2_vals[top_idx], s=20, alpha=0.9, color='crimson', lw=0,
               label=f'top {n_ensemble_keep}' if i == 0 else None)

ax.set_xticks(np.arange(len(setting_labels)))
ax.set_xticklabels(setting_labels, rotation=45, ha='right', fontsize=7)
ax.set_ylim(0, 1)
ax.set_ylabel('R² test')
ax.set_xlabel('Hyperparameter setting')
ax.set_title(f'R² distribution per hyperparameter set (npc={npc_fixed})')
ax.legend(frameon=False, fontsize=7, loc='best')
ax.grid(axis='y', alpha=0.2)
fig.tight_layout()

if input('save? ').strip().lower() == 'save':
    fig.savefig(f'./figures/SI_tara_hparams_r2scatter_npc{npc_fixed}.svg', bbox_inches='tight')

## Pairwise confusion matrices

In [ ]:
n_sets = len(assignment_lists)

conf_mats = {}
vmax = 0
for a in range(n_sets):
    for b in range(n_sets):
        ids_a = assignment_lists[a][spec_keep]
        ids_b = assignment_lists[b][spec_keep]

        ua = np.unique(ids_a)
        ub = np.unique(ids_b)

        mat = np.zeros((len(ua), len(ub)), dtype=int)
        for i, ga in enumerate(ua):
            for j, gb in enumerate(ub):
                mat[i, j] = np.sum((ids_a == ga) & (ids_b == gb))

        conf_mats[(a, b)] = mat
        vmax = max(vmax, mat.max())

fig, ax = plt.subplots(n_sets, n_sets, figsize=(0.5 * n_sets, 0.5 * n_sets), dpi=200)
if n_sets == 1:
    ax = np.array([[ax]])

for a in range(n_sets):
    for b in range(n_sets):
        this_ax = ax[a, b]
        if a > b:
            this_ax.axis('off')
            continue

        mat = conf_mats[(a, b)]
        this_ax.imshow(mat.T, cmap='binary', vmin=0, vmax=vmax, interpolation='nearest')

        for i in range(npc_fixed):
            for j in range(npc_fixed):
                color = 'white' if mat[j, i] > vmax / 2 else 'k'
                this_ax.text(i, j, f'{mat[j, i]}', color=color,
                             ha='center', va='center', fontsize=6)

        this_ax.set_xticks([])
        this_ax.set_yticks([])
        if a == 0:
            this_ax.set_title(f'{b}', fontsize=6, pad=2)
        if b == 0:
            this_ax.set_ylabel(f'{a}', fontsize=6, rotation=0, labelpad=8, va='center')

fig.suptitle('Pairwise confusion matrices across hyperparameter settings', fontsize=9, y=0.995)
fig.tight_layout(pad=0.2)

for i, lbl in enumerate(setting_labels_kept):
    print(i, lbl)

if input('save? ').strip().lower() == 'save':
    fig.savefig(f'./figures/SI_tara_hparams_confusion_npc{npc_fixed}.svg', bbox_inches='tight')